# 03 - Sharpness robustness test (local GPU)

**Question:** test frames are sharper than train frames (sharpness 9.3 vs 6.5). Which of our feature groups break when frames get sharper (or blurrier)?

**Method:** take 150 held-out groups (300 clips, fold 0). Train each feature group's model on the other folds with CLEAN frames.
Then evaluate the 300 clips three ways: clean, **sharpened** (to mimic test), **blurred** (the other direction). Frames are re-embedded with DINOv2 on the local GPU (about 5 min).
We look at (a) how much AUC drops, and (b) how much the average prediction shifts (a shifted score = the feature reads sharpness as 'crash').

In [1]:
import sys, os, time
sys.path.append("..")
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFilter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from transformers import AutoModel
from src.common import ART, DATA, load_labels, make_folds

device = "cuda" if torch.cuda.is_available() else "cpu"
lab = load_labels().merge(make_folds(), on="clip_id")
emb = np.load(ART / "features/dinov2b_train.npy").astype(np.float32)   # clean embeddings of all train clips
y, fold = lab["label"].values, lab["fold"].values

# 150 whole groups from fold 0 (both twins kept), used as the shift-test set
groups0 = lab[lab.fold == 0].group_id.unique()[:150]
test_idx = np.where(lab.group_id.isin(groups0))[0]
train_idx = np.where(fold != 0)[0]
print("shift-test clips:", len(test_idx), "| train clips:", len(train_idx), "| device:", device)

shift-test clips: 300 | train clips: 2400 | device: cuda


## 1. Make frames sharper / blurrier
We need a sharpness measure (the same one as in the EDA: average edge strength of the middle frame) and two image filters.
We pick filter strengths so that **sharpened train frames reach test-like sharpness (about 9.3)**.

In [2]:
def sharpness(img):
    g = np.asarray(img.convert("L"), dtype=np.float32)
    gy, gx = np.gradient(g)
    return np.sqrt(gx**2 + gy**2).mean()

def sharpen(img, percent):  return img.filter(ImageFilter.UnsharpMask(radius=1, percent=percent, threshold=0))
def blur(img, radius):      return img.filter(ImageFilter.GaussianBlur(radius))

ids = lab.clip_id.values[test_idx[:40]]
frames = [Image.open(f"{DATA}/train/{c}/frame_015.jpg").convert("RGB") for c in ids]
print("clean           :", np.mean([sharpness(f) for f in frames]).round(2), "(train average is 6.5, test is 9.3)")
for p in [50, 100, 150]:
    print(f"sharpen {p:3d}%   :", np.mean([sharpness(sharpen(f, p)) for f in frames]).round(2))
for r in [1.0, 1.5]:
    print(f"blur radius {r}  :", np.mean([sharpness(blur(f, r)) for f in frames]).round(2))

clean           : 6.61 (train average is 6.5, test is 9.3)


sharpen  50%   : 7.91
sharpen 100%   : 9.4


sharpen 150%   : 10.61
blur radius 1.0  : 4.43


blur radius 1.5  : 3.58


Radius 1 and 100% gives about 9.4 (test-like), so we use that; blur radius 1.5 is our 'blurrier than train' case.

In [3]:
SHARPEN_PERCENT = 100      # radius 1 + 100% gives sharpness about 9.4 = test-like
BLUR_RADIUS = 1.5

## 2. Re-embed the 300 clips with the filter applied
Same model and settings as `01_extract_features` (CLS + mean patch, 1536 numbers per frame).

In [4]:
model = AutoModel.from_pretrained("facebook/dinov2-base").eval().to(device)
if device == "cuda":
    model = model.half()
H, W = 224, 392
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def embed_clips(clip_ids, transform):
    out = []
    for cid in tqdm(clip_ids):
        frames = []
        for j in range(30):
            img = Image.open(f"{DATA}/train/{cid}/frame_{j:03d}.jpg").convert("RGB")
            img = transform(img).resize((W, H), Image.BICUBIC)       # filter first, then resize like before
            frames.append(np.asarray(img))
        x = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2).to(device).float() / 255
        x = ((x - mean) / std).half() if device == "cuda" else (x - mean) / std
        tokens = model(pixel_values=x).last_hidden_state
        out.append(torch.cat([tokens[:, 0], tokens[:, 1:].mean(1)], -1).float().cpu().numpy())
    return np.stack(out)                                             # (clips, 30, 1536)

shift_ids = lab.clip_id.values[test_idx]
t0 = time.time()
emb_sharp = embed_clips(shift_ids, lambda im: sharpen(im, SHARPEN_PERCENT))
emb_blur = embed_clips(shift_ids, lambda im: blur(im, BLUR_RADIUS))
emb_clean_check = embed_clips(shift_ids[:20], lambda im: im)         # sanity: re-embedding clean frames should match Kaggle's features
print(f"{time.time() - t0:.0f}s | clean re-embed vs Kaggle features, mean abs diff:",
      np.abs(emb_clean_check - emb[test_idx[:20]]).mean().round(4))

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

289s | clean re-embed vs Kaggle features, mean abs diff: 0.0014


## 3. Same feature groups as the baseline, then train clean / test clean vs sharpened vs blurred

In [5]:
def make_groups(e):
    d1 = np.abs(e[:, 1:] - e[:, :-1])
    step2 = np.linalg.norm(e[:, 2:] - e[:, :-2], axis=-1)
    shape = step2 / (step2.mean(1, keepdims=True) + 1e-6)
    return {"mean over time": e.mean(1), "std over time": e.std(1), "max over time": e.max(1),
            "diff mean": d1.mean(1), "diff max": d1.max(1),
            "curve raw (28)": step2, "curve shape (28)": shape}

g_train = make_groups(emb[train_idx])
g_clean = make_groups(emb[test_idx])
g_sharp = make_groups(emb_sharp)
g_blur = make_groups(emb_blur)
y_test = y[test_idx]

rows = []
for name in g_train:
    model_lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=2000)).fit(g_train[name], y[train_idx])
    row = {"group": name}
    for tag, g in [("clean", g_clean), ("sharp", g_sharp), ("blur", g_blur)]:
        p = model_lr.predict_proba(g[name])[:, 1]
        row[f"AUC {tag}"] = roc_auc_score(y_test, p)
        row[f"mean pred {tag}"] = p.mean()
    rows.append(row)
res = pd.DataFrame(rows).set_index("group").round(3)
res["AUC drop (sharp)"] = (res["AUC clean"] - res["AUC sharp"]).round(3)
res["pred shift (sharp)"] = (res["mean pred sharp"] - res["mean pred clean"]).round(3)
res

,AUC clean,mean pred clean,AUC sharp,mean pred sharp,AUC blur,mean pred blur,AUC drop (sharp),pred shift (sharp)
group,,,,,,,,
mean over time,0.888,0.511,0.901,0.594,0.790,0.549,-0.013,0.083
std over time,0.880,0.518,0.882,0.536,0.779,0.580,-0.002,0.018
max over time,0.905,0.511,0.912,0.583,0.805,0.561,-0.007,0.072
diff mean,0.764,0.517,0.765,0.539,0.714,0.511,-0.001,0.022
diff max,0.735,0.510,0.743,0.524,0.690,0.494,-0.008,0.014
curve raw (28),0.918,0.506,0.913,0.509,0.901,0.522,0.005,0.003
curve shape (28),0.917,0.507,0.917,0.508,0.896,0.498,0.000,0.001


## How to read the table
- **AUC drop (sharp)** small = the feature group still ranks clips correctly when frames are sharper (good).
- **pred shift (sharp)** large = the model's average score moves just because of sharpness. Since all test clips are sharp it is a uniform shift (AUC may survive),
  but if the shift is not uniform across clips, ranking breaks.
- Prefer groups with a small drop in BOTH directions (sharp and blur).